In [10]:
%run /Users/manojrammopati/Downloads/bupa_insurance_project/01_Bronze_Layer/Notebooks/_01_bronze_container_connect.ipynb

25/11/25 00:44:08 WARN Utils: Your hostname, Manojrams-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 172.19.98.217 instead (on interface en0)
25/11/25 00:44:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/manojrammopati/.ivy2/cache
The jars for the packages stored in: /Users/manojrammopati/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-azure added as a dependency
com.azure#azure-storage-blob added as a dependency
com.azure#azure-identity added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f069c2f0-3659-494b-be6e-f31820762eb0;1.0
	confs: [default]


:: loading settings :: url = jar:file:/opt/anaconda3/envs/spark_local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-azure;3.3.4 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found com.microsoft.azure#azure-storage;7.0.1 in central
	found com.fasterxml.jackson.core#jackson-core;2.12.7 in central
	found org.slf4j#slf4j-api;1.7.36 in central
	found com.microsoft.azure#azure-keyvault-core;1.0.0 in central
	found com.google.guava#guava;27.0-jre in central
	found com.google.guava#failureaccess;1.0 in central
	found com.google.guava#listenablefuture;9999.0-empty-to-avoid-conflict-with-guava in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found org.checkerframework#checker-qual;2.5.2 in central
	found com.google.errorpron

Preconnectors ran successfully and connected to 'clientdatastorage' storage accounts
 'policy_df, claim_df, member_df, provider_df' are ready to use from container 'raw data'(meridian architecture of bronze) 


In [33]:
import sys
import importlib
import utils_silver
from utils_silver import *

# Add the Silver layer directory to sys.path and import the utils module

silver_dir = "/Users/manojrammopati/Downloads/bupa_insurance_project/02_Silver_Layer"
if silver_dir not in sys.path:
    sys.path.insert(0, silver_dir)

# Now import (and reload to pick up edits)
importlib.reload(utils_silver)


paths_bronze, paths_silver = utils_silver.build_paths(storage_account1)
print(sorted(paths_silver.keys()))
# Optionally bring helper names into the notebook namespace (avoid if you prefer module namespace)

print("Imported utils_silver from", utils_silver.__file__)

✅ utils_silver.py loaded
['_metrics', '_quarantine', '_ref_dim_channel', '_ref_dim_product_line', '_reference', 'claims', 'members', 'policies', 'providers']
Imported utils_silver from /Users/manojrammopati/Downloads/bupa_insurance_project/02_Silver_Layer/utils_silver.py


In [23]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime



# Build paths from utils
paths_bronze, paths_silver = build_paths(storage_account1)

print("✅ Bronze paths loaded")
print(paths_bronze)
print("✅ Silver paths loaded")
print(paths_silver)


✅ Bronze paths loaded
{'policies': 'abfss://rawdata@clientdatastorage.dfs.core.windows.net/policies', 'members': 'abfss://rawdata@clientdatastorage.dfs.core.windows.net/members', 'claims': 'abfss://rawdata@clientdatastorage.dfs.core.windows.net/claims', 'providers': 'abfss://rawdata@clientdatastorage.dfs.core.windows.net/providers'}
✅ Silver paths loaded
{'policies': 'abfss://silverdata@clientdatastorage.dfs.core.windows.net/policies', 'members': 'abfss://silverdata@clientdatastorage.dfs.core.windows.net/members', 'claims': 'abfss://silverdata@clientdatastorage.dfs.core.windows.net/claims', 'providers': 'abfss://silverdata@clientdatastorage.dfs.core.windows.net/providers', '_reference': 'abfss://silverdata@clientdatastorage.dfs.core.windows.net/reference', '_quarantine': 'abfss://silverdata@clientdatastorage.dfs.core.windows.net/quarantine', '_metrics': 'abfss://silverdata@clientdatastorage.dfs.core.windows.net/runmetrics'}


In [24]:
BRONZE_POLICIES_PATH = paths_bronze["policies"]
print("Reading bronze policies from:", BRONZE_POLICIES_PATH)

pol_bz = spark.read.format("delta").load(BRONZE_POLICIES_PATH)

pol = (pol_bz.select(
        F.col("Policy_ID").cast("string"),
        F.col("Customer_ID").cast("string"),
        F.col("Product_Line").cast("string"),
        F.col("Sum_Insured_GBP").cast("double"),
        F.col("Annual_Premium_GBP").cast("double"),
        F.to_date("Policy_Start_Date").alias("Policy_Start_Date"),
        F.to_date("Policy_End_Date").alias("Policy_End_Date"),
        F.col("Renewal_Offered_Flag").cast("int"),
        F.col("Renewal_Accepted_Flag").cast("int"),
        F.col("Discount_Offered_%").cast("double"),
        F.col("Channel").cast("string")
     ))

print(f"Bronze rowcount: {pol.count()}")
pol.show(5, truncate=False)


Reading bronze policies from: abfss://rawdata@clientdatastorage.dfs.core.windows.net/policies
Bronze rowcount: 381109


+----------+------------+------------+------------------+------------------+-----------------+---------------+--------------------+---------------------+------------------+-------+
|Policy_ID |Customer_ID |Product_Line|Sum_Insured_GBP   |Annual_Premium_GBP|Policy_Start_Date|Policy_End_Date|Renewal_Offered_Flag|Renewal_Accepted_Flag|Discount_Offered_%|Channel|
+----------+------------+------------+------------------+------------------+-----------------+---------------+--------------------+---------------------+------------------+-------+
|P_00000001|CUST_0000001|Dental      |2193748.339982436 |40454.0           |2025-10-12       |2026-05-05     |1                   |0                    |3.418994275911136 |Partner|
|P_00000002|CUST_0000002|Health      |1908192.4592739474|33536.0           |2025-06-04       |2026-08-23     |1                   |1                    |16.10228791601092 |Partner|
|P_00000003|CUST_0000003|Health      |2356477.1034685415|38294.0           |2022-01-18       |2

In [25]:
policy_schema = StructType([
    StructField("Policy_ID", StringType()),
    StructField("Customer_ID", StringType()),
    StructField("Product_Line", StringType()),
    StructField("Sum_Insured_GBP", DoubleType()),
    StructField("Annual_Premium_GBP", DoubleType()),
    StructField("Policy_Start_Date", DateType()),
    StructField("Policy_End_Date", DateType()),
    StructField("Renewal_Offered_Flag", IntegerType()),
    StructField("Renewal_Accepted_Flag", IntegerType()),
    StructField("Discount_Offered_%", DoubleType()),
    StructField("Channel", StringType())
])

pol = enforce_schema(pol, policy_schema)

pol_bad = pol.filter("Policy_ID IS NULL OR Customer_ID IS NULL")
pol_ok  = pol.filter("Policy_ID IS NOT NULL AND Customer_ID IS NOT NULL")

if pol_bad.take(1):
    quarantine(pol_bad, "null_key_policy_or_customer", "policies", paths_silver)

print(f"After key check: {pol_ok.count()} rows")


After key check: 381109 rows


In [26]:
# Fix reversed start/end dates safely
pol_ok = fix_dates(pol_ok, "Policy_Start_Date", "Policy_End_Date")

# Soft DQ flags (keep rows, just tag them)
pol_ok = pol_ok.withColumn(
    "dq_money_valid",
    F.when((F.col("Annual_Premium_GBP") >= 0) & (F.col("Sum_Insured_GBP") >= 0), 1).otherwise(0)
).withColumn(
    "dq_discount_valid",
    F.when(F.col("Discount_Offered_%").between(0, 100) | F.col("Discount_Offered_%").isNull(), 1).otherwise(0)
).withColumn(
    "dq_renewal_valid",
    F.when(
        ~((F.col("Renewal_Accepted_Flag") == 1) &
          (F.coalesce(F.col("Renewal_Offered_Flag"), F.lit(0)) != 1)),
        1
    ).otherwise(0)
)


In [27]:
# Latest record wins per Policy_ID
pol_silver = drop_dupes_keep_latest(pol_ok, ["Policy_ID"], ["Policy_Start_Date"])

# Normalize categories
pol_silver = (pol_silver
    .withColumn("Product_Line", F.initcap(F.trim("Product_Line")))
    .withColumn("Channel", F.initcap(F.trim("Channel")))
)

# Derived fields
pol_silver = pol_silver.withColumn(
    "Policy_Duration_Days",
    F.when(
        F.col("Policy_End_Date").isNotNull() & F.col("Policy_Start_Date").isNotNull(),
        F.datediff("Policy_End_Date", "Policy_Start_Date")
    )
)

pol_silver = pol_silver.withColumn(
    "Renewal_Conversion",
    F.when(F.col("Renewal_Offered_Flag") == 1, F.col("Renewal_Accepted_Flag"))
)

pol_silver.show(10, truncate=False)


+----------+------------+------------+------------------+------------------+-----------------+---------------+--------------------+---------------------+------------------+-------+--------------+-----------------+----------------+--------------------+------------------+
|Policy_ID |Customer_ID |Product_Line|Sum_Insured_GBP   |Annual_Premium_GBP|Policy_Start_Date|Policy_End_Date|Renewal_Offered_Flag|Renewal_Accepted_Flag|Discount_Offered_%|Channel|dq_money_valid|dq_discount_valid|dq_renewal_valid|Policy_Duration_Days|Renewal_Conversion|
+----------+------------+------------+------------------+------------------+-----------------+---------------+--------------------+---------------------+------------------+-------+--------------+-----------------+----------------+--------------------+------------------+
|P_00000006|CUST_0000006|Dental      |167095.0764838565 |2630.0            |2025-09-23       |2026-10-22     |1                   |0                    |4.187409955154    |Partner|1      

In [34]:
dim_channel_path = paths_silver["_ref_dim_channel"]
dim_product_path = paths_silver["_ref_dim_product_line"]

dim_channel = spark.createDataFrame(
    [("Agent",), ("Broker",), ("Online",), ("Partner",)],
    "Channel STRING"
)

dim_product_line = spark.createDataFrame(
    [("Accident",), ("Dental",), ("Health",), ("Motor",), ("Travel",)],
    "Product_Line STRING"
)

dim_channel.write.format("delta").mode("overwrite").save(dim_channel_path)
dim_product_line.write.format("delta").mode("overwrite").save(dim_product_path)

print("✅ dim_channel written to:", dim_channel_path)
print("✅ dim_product_line written to:", dim_product_path)


✅ dim_channel written to: abfss://silverdata@clientdatastorage.dfs.core.windows.net/reference/dim_channel
✅ dim_product_line written to: abfss://silverdata@clientdatastorage.dfs.core.windows.net/reference/dim_product_line


In [35]:
# Normalize renewal flags to safe 0/1 ints
def to_flag(colname):
    s = F.lower(F.trim(F.col(colname).cast("string")))
    return F.when(s.isin("1","y","yes","true","t","on"), 1) \
            .when(s.isin("0","n","no","false","f","off"), 0) \
            .otherwise(F.col(colname).cast("int"))

pol_silver = (pol_silver
    .withColumn("Renewal_Offered_Flag",  to_flag("Renewal_Offered_Flag"))
    .withColumn("Renewal_Accepted_Flag", to_flag("Renewal_Accepted_Flag"))
)

# --- Core DQ rules ---
dq_expect(pol_silver, "key_not_null",
          "Policy_ID IS NOT NULL AND Customer_ID IS NOT NULL",
          "error", "policies", paths_silver)

dq_expect(pol_silver, "date_order",
          "Policy_Start_Date IS NULL OR Policy_End_Date IS NULL OR Policy_End_Date >= Policy_Start_Date",
          "error", "policies", paths_silver)

dq_expect(pol_silver, "money_non_negative",
          "coalesce(Annual_Premium_GBP,0) >= 0 AND coalesce(Sum_Insured_GBP,0) >= 0",
          "error", "policies", paths_silver)

dq_expect(pol_silver, "discount_bounds",
          "`Discount_Offered_%` IS NULL OR (`Discount_Offered_%` BETWEEN 0 AND 100)",
          "error", "policies", paths_silver)

# --- Renewal SLA gate ---
viol_df = pol_silver.filter(
    (F.col("Renewal_Accepted_Flag") == 1) &
    (F.coalesce(F.col("Renewal_Offered_Flag"), F.lit(0)) != 1)
)
viol_cnt = viol_df.count()
total    = pol_silver.count()
viol_pct = 0 if total == 0 else round(viol_cnt/total*100.0, 4)

if viol_cnt > 0:
    quarantine(viol_df, "renewal_inconsistent", "policies", paths_silver)

print(f"[RECON] renewal_inconsistent: {viol_cnt}/{total} ({viol_pct}%)")

SLA_PCT = 25.0
if viol_pct > SLA_PCT:
    raise Exception(f"DQ gate failed: policies · renewal_consistency {viol_pct}% > {SLA_PCT}%")
else:
    print("✅ DQ PASS renewal_consistency (within SLA)")

# --- Dictionary checks ---
dim_channel      = spark.read.format("delta").load(dim_channel_path)
dim_product_line = spark.read.format("delta").load(dim_product_path)

dq_left_anti_ref(pol_silver, dim_channel, "Channel", "Channel",
                 "channel_in_dictionary", "error", "policies", paths_silver)

dq_left_anti_ref(pol_silver, dim_product_line, "Product_Line", "Product_Line",
                 "product_line_in_dictionary", "error", "policies", paths_silver)


✅ DQ PASS [policies] key_not_null


✅ DQ PASS [policies] date_order


✅ DQ PASS [policies] money_non_negative


✅ DQ PASS [policies] discount_bounds


25/11/25 01:22:18 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


[QUARANTINE] renewal_inconsistent → abfss://silverdata@clientdatastorage.dfs.core.windows.net/_quarantine/policies/renewal_inconsistent (rows=66981)
[RECON] renewal_inconsistent: 66981/381109 (17.5753%)
✅ DQ PASS renewal_consistency (within SLA)


✅ DQ PASS [policies] channel_in_dictionary


✅ DQ PASS [policies] product_line_in_dictionary


In [36]:
SILVER_POLICIES_PATH = paths_silver["policies"]

pol_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(SILVER_POLICIES_PATH)

print("✅ Policies Silver written to:", SILVER_POLICIES_PATH)


25/11/25 01:24:58 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


✅ Policies Silver written to: abfss://silverdata@clientdatastorage.dfs.core.windows.net/policies


In [37]:
write_metric("rowcount_policies_silver", pol_silver.count(), "policies_silver", paths_silver)
write_metric("distinct_policy_ids", pol_silver.select("Policy_ID").distinct().count(), "policies_silver", paths_silver)

metrics_df = spark.read.format("delta").load(paths_silver["_metrics"])
metrics_df.orderBy(F.col("ts").desc()).show(truncate=False)


NameError: name 'spark' is not defined

In [38]:
from pyspark.sql import functions as F
from datetime import datetime

# ---- Metrics writer (path-based, no metastore needed) ----
def write_metric(name: str, value, context: str, paths_silver: dict):
    metrics_path = paths_silver["_metrics"]

    metric_row = spark.createDataFrame(
        [(name, str(value), context, datetime.utcnow())],
        "metric STRING, value STRING, context STRING, ts TIMESTAMP"
    )

    (metric_row.write
        .format("delta")
        .mode("append")
        .save(metrics_path)
    )

    print(f"✅ metric written to {metrics_path} → {name}={value}")

# ---- Write run metrics ----
write_metric("rowcount_policies_silver", pol_silver.count(), "policies_silver", paths_silver)
write_metric("distinct_policy_ids", pol_silver.select("Policy_ID").distinct().count(), "policies_silver", paths_silver)

# ---- Read back + display ----
metrics_df = spark.read.format("delta").load(paths_silver["_metrics"])
metrics_df.orderBy(F.col("ts").desc()).show(truncate=False)


✅ metric written to abfss://silverdata@clientdatastorage.dfs.core.windows.net/_metrics → rowcount_policies_silver=381109


✅ metric written to abfss://silverdata@clientdatastorage.dfs.core.windows.net/_metrics → distinct_policy_ids=381109


+------------------------+------+---------------+--------------------------+
|metric                  |value |context        |ts                        |
+------------------------+------+---------------+--------------------------+
|distinct_policy_ids     |381109|policies_silver|2025-11-25 01:31:29.412102|
|rowcount_policies_silver|381109|policies_silver|2025-11-25 01:31:06.610274|
+------------------------+------+---------------+--------------------------+



In [39]:
pol_sv_check = spark.read.format("delta").load(paths_silver["policies"])

print("Silver policies count:", pol_sv_check.count())
pol_sv_check.show(10, truncate=False)

print("✅ SUCCESS: policies_silver completed")


Silver policies count: 381109


+----------+------------+------------+------------------+------------------+-----------------+---------------+--------------------+---------------------+------------------+-------+--------------+-----------------+----------------+--------------------+------------------+
|Policy_ID |Customer_ID |Product_Line|Sum_Insured_GBP   |Annual_Premium_GBP|Policy_Start_Date|Policy_End_Date|Renewal_Offered_Flag|Renewal_Accepted_Flag|Discount_Offered_%|Channel|dq_money_valid|dq_discount_valid|dq_renewal_valid|Policy_Duration_Days|Renewal_Conversion|
+----------+------------+------------+------------------+------------------+-----------------+---------------+--------------------+---------------------+------------------+-------+--------------+-----------------+----------------+--------------------+------------------+
|P_00000003|CUST_0000003|Health      |2356477.1034685415|38294.0           |2022-01-18       |2023-01-07     |1                   |1                    |11.551851153178037|Partner|1      